# Điều hướng notebook
Runtime được cấu hình qua `LAB09_RUNTIME`, mặc định `C:/Users/tasdus_117/Desktop/NLP_runtime/lab09`. Notebook dùng fast tokenizer, kiểm tra dữ liệu thật và giải thích first-subtoken. Mọi bảng phụ được ghi ngoài thư mục bài nộp.

[3. Dataset: CONLL-2003](#lab09-section-3) · [8. Subword Tokenization](#lab09-section-8) · [9. Label Alignment](#lab09-section-9) · [10. Alignment Strategies](#lab09-section-10) · [11. Ignore Index](#lab09-section-11) · [16. Core Implementation](#lab09-section-16) · [17. Experiment 1: Tokenization Analysis](#lab09-section-17)

### Cách đọc và chạy notebook

Chạy từ trên xuống trong kernel sạch, cwd=lab09. Dữ liệu tải bởi `baseline_ner.py prepare`, không chạy script dataset từ xa.
<a id="lab09-section-8"></a>
## 8. Subword Tokenization

[Bảng words → subwords](#word-subword-table).

<a id="lab09-section-9"></a>
## 9. Label Alignment

[Bảng word_ids → aligned labels](#word-subword-table); `information_extraction.encode_rows()` và `baseline_ner.align_labels()`.

<a id="lab09-section-10"></a>
## 10. Alignment Strategies

Chiến lược first-subtoken: chỉ mảnh đầu một word có gold label. <a id="lab09-section-11"></a>
## 11. Ignore Index

Special/pad/continuation = -100. Attention mask vẫn là 1 cho continuation: nó tham gia contextual attention, chỉ không có target loss trực tiếp.


In [1]:
import os
os.environ['USE_TF'] = '0'
os.environ['HF_HUB_OFFLINE'] = '1'
import json
from collections import Counter
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer, DataCollatorForTokenClassification
from baseline_ner import ROOT, ART, CACHE, LABELS, load_data, align_labels, chunk_words, write_json
data = load_data(inspect=False)
tokenizer = AutoTokenizer.from_pretrained(CACHE / 'backbone', use_fast=True, local_files_only=True)
assert tokenizer.is_fast
display(pd.DataFrame(json.loads((ART / 'dataset_inspection.json').read_text())['splits']).T)
print('label mapping:', dict(enumerate(LABELS)))
print('sample:', data['train'][0])
print('sources:', json.loads((ART / 'sources.json').read_text()))

,sentences,words,iob_to_iob2_changes
train,14041,203621,0
validation,3250,51362,0
test,3453,46435,0


label mapping: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}
sample: {'id': 0, 'source_id': '0', 'words': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'], 'original_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}
sources: {'dataset': {'repo': 'eriktks/conll2003', 'revision': 'ce85b39f9dd99f552d0739d456814e95fb6a39b0', 'files': [{'name': 'conll2003/test/0000.parquet', 'url': 'https://huggingface.co/datasets/eriktks/conll2003/resolve/ce85b39f9dd99f552d0739d456814e95fb6a39b0/conll2003/test/0000.parquet', 'bytes': 283307, 'sha256': '14f894263b4ee3563395a57b8ba4cfaabec7db4ea956ba34e2f84a19781cdc43'}, {'name': 'conll2003/train/0000.parquet', 'url': 'https://huggingface.co/datasets/eriktks/conll2003/resolve/ce85b39f9dd99f552d0739d456814e95fb6a39b0/conll2003/train/0000.parquet', 'bytes': 1227788, 'sha256': '546cbb4200

<a id="lab09-section-3"></a>
## 3. Dataset: CONLL-2003

Dataset card phân biệt IOB1 gốc với bản IOB2 trên Hugging Face. `load_data(inspect=False)` kiểm tra mapping metadata Parquet và đếm số I-X phải đổi thành B-X khi đứng sau O/type khác. Bảng trên ghi số đổi từng split; mọi code dùng cùng normalization IOB2. Không suy luận scheme chỉ từ tên CoNLL.
[Dataset card](https://huggingface.co/datasets/eriktks/conll2003), [token classification API](https://huggingface.co/docs/transformers/tasks/token_classification).


In [2]:
# Chọn 40 câu validation: 20 đầu <=20 words và 20 đầu >20 words.
# Không dùng tokenization experiment để chọn checkpoint/hyperparameters.
short = [r for r in data['validation'] if len(r['words']) <= 20][:20]
long = [r for r in data['validation'] if len(r['words']) > 20][:20]
selected = short + long
assert len(selected) == 40
sentences, word_rows = [], []
for row in selected:
    encoded = tokenizer(row['words'], is_split_into_words=True, truncation=False)
    counts = Counter(w for w in encoded.word_ids() if w is not None)
    assert set(counts) == set(range(len(row['words'])))
    total = sum(counts.values())
    sentences.append(dict(split='validation', id=row['id'], words=len(row['words']), subword_tokens=total,
        split_words=sum(n > 1 for n in counts.values()), extra_pieces=total-len(row['words']),
        R=total/len(row['words']), group='short' if len(row['words']) <= 20 else 'long'))
    for wid, word in enumerate(row['words']):
        pieces = [t for t, w in zip(encoded.tokens(), encoded.word_ids()) if w == wid]
        word_rows.append(dict(id=row['id'], word=word, gold=row['tags'][wid], pieces=' '.join(pieces),
            subwords=len(pieces), split=int(len(pieces)>1), extra=len(pieces)-1,
            group='non-entity' if row['tags'][wid]=='O' else 'entity'))
sentence_df, word_df = pd.DataFrame(sentences), pd.DataFrame(word_rows)
sentence_df.to_csv(ART / 'tokenization_analysis.csv', index=False)
word_df.to_csv(ART / 'tokenization_words.csv', index=False)
display(sentence_df)
sentence_summary = sentence_df.groupby('group')[['words','subword_tokens','split_words','extra_pieces']].sum()
sentence_summary['R'] = sentence_summary.subword_tokens / sentence_summary.words
display(sentence_summary)
word_summary = word_df.groupby('group').agg(words=('word','size'), subword_tokens=('subwords','sum'), split_words=('split','sum'), extra_pieces=('extra','sum'))
word_summary['R'] = word_summary.subword_tokens / word_summary.words
word_summary['fraction_words_split'] = word_summary.split_words / word_summary.words
display(word_summary)
display(word_df[word_df['subwords']>1].drop_duplicates('word').sort_values('subwords',ascending=False).head(20))
write_json(ART / 'tokenization_summary.json', dict(sentences=len(selected), short=sentence_summary.reset_index().to_dict('records'), word_groups=word_summary.reset_index().to_dict('records')))

,split,id,words,subword_tokens,split_words,extra_pieces,R,group
0,validation,0,11,35,8,24,3.181818,short
1,validation,1,2,9,2,7,4.500000,short
2,validation,15,9,11,1,2,1.222222,short
3,validation,16,7,24,5,17,3.428571,short
4,validation,17,2,9,2,7,4.500000,short
5,validation,18,14,16,1,2,1.142857,short
6,validation,19,12,12,0,0,1.000000,short
7,validation,20,13,17,3,4,1.307692,short
8,validation,21,7,7,0,0,1.000000,short
9,validation,23,18,28,6,10,1.555556,short


,words,subword_tokens,split_words,extra_pieces,R
group,,,,,
long,626,750,76,124,1.198083
short,171,274,40,103,1.602339


,words,subword_tokens,split_words,extra_pieces,R,fraction_words_split
group,,,,,,
entity,169,271,56,102,1.603550,0.331361
non-entity,628,753,60,125,1.199045,0.095541


,id,word,gold,pieces,subwords,split,extra,group
2,0,LEICESTERSHIRE,B-ORG,L ##EI ##CE ##ST ##ER ##S ##H ##IR ##E,9,1,8,entity
26,16,CHAMPIONSHIP,I-MISC,CH ##AM ##PI ##ON ##S ##H ##IP,7,1,6,entity
99,29,INTINERARY,O,IN ##TI ##NE ##RA ##R ##Y,6,1,5,non-entity
718,27,T.O'Gorman,B-PER,T . O ' Go ##rman,6,1,5,entity
24,16,ENGLISH,B-MISC,E ##NG ##L ##IS ##H,5,1,4,entity
12,1,1996-08-30,O,1996 - 08 - 30,5,1,4,non-entity
589,22,Chester-le-Street,B-LOC,Chester - le - Street,5,1,4,entity
758,79,26-year-old,O,26 - year - old,5,1,4,non-entity
11,1,LONDON,B-LOC,L ##ON ##D ##ON,4,1,3,entity
0,0,CRICKET,O,CR ##IC ##KE ##T,4,1,3,non-entity


<a id="lab09-section-17"></a>
## 17. Experiment 1: Tokenization Analysis

Words là số tokens trong annotation gốc. Subword tokens đếm mọi mảnh có word_id, loại special/pad. Split words đếm số words có >1 mảnh. Extra pieces = tổng (số mảnh − 1). R = subwords/words, là hệ số nở độ dài, **không phải phần trăm words bị split**. Tỷ lệ words bị split = split_words/words.
Ngắn <=20, dài >20 words; nhóm entity gồm B/I, non-entity gồm O. Các số là aggregate có trọng số word, không phải trung bình các R câu. Mẫu có chủ đích 20+20, không đại diện toàn corpus. Bảng ví dụ là từ thật trong 40 câu, chỉ nêu quan sát về từ dài/tên riêng, không kết luận causal về lỗi NER.

<a id="word-subword-table"></a>


In [3]:
# Một câu thật có word bị chia; thêm 3 padding tokens để nhìn trực tiếp.
sample_id = next(r['id'] for r in selected if any(w['id']==r['id'] and w['subwords']>1 for w in word_rows))
sample = next(r for r in selected if r['id']==sample_id)
base = tokenizer(sample['words'], is_split_into_words=True)
enc = tokenizer(sample['words'], is_split_into_words=True, padding='max_length', max_length=len(base['input_ids'])+3, truncation=False)
wids = enc.word_ids()
aligned = align_labels(wids, [LABELS.index(t) for t in sample['tags']])
display(pd.DataFrame({'word':sample['words'], 'gold_BIO':sample['tags']}))
table = pd.DataFrame(dict(subword=enc.tokens(), input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], word_ids=wids,
    word=[sample['words'][w] if w is not None else None for w in wids],
    gold_BIO=[sample['tags'][w] if w is not None else None for w in wids], aligned_id=aligned,
    aligned_label=[LABELS[t] if t != -100 else 'IGNORE' for t in aligned]))
display(table)
seen = set()
for i, wid in enumerate(wids):
    if wid is None or wid in seen:
        assert aligned[i] == -100
    else:
        assert aligned[i] == LABELS.index(sample['tags'][wid])
    if wid is not None and wid in seen:
        assert enc['attention_mask'][i] == 1
    seen.add(wid)
assert sum(t != -100 for t in aligned) == len(sample['words'])
assert any(a == 0 and y == -100 for a, y in zip(enc['attention_mask'], aligned))
assert any(wids[i] is not None and wids[i] == wids[i-1] for i in range(1,len(wids)))
print('PASS: split word, special tokens, padding, first-subtoken')

,word,gold_BIO
0,CRICKET,O
1,-,O
2,LEICESTERSHIRE,B-ORG
3,TAKE,O
4,OVER,O
5,AT,O
6,TOP,O
7,AFTER,O
8,INNINGS,O
9,VICTORY,O


,subword,input_ids,attention_mask,word_ids,word,gold_BIO,aligned_id,aligned_label
0,[CLS],101,1,NaN,NaN,NaN,-100,IGNORE
1,CR,15531,1,0.0,CRICKET,O,0,O
2,##IC,9741,1,0.0,CRICKET,O,-100,IGNORE
3,##KE,22441,1,0.0,CRICKET,O,-100,IGNORE
4,##T,1942,1,0.0,CRICKET,O,-100,IGNORE
5,-,118,1,1.0,-,O,0,O
6,L,149,1,2.0,LEICESTERSHIRE,B-ORG,3,B-ORG
7,##EI,27514,1,2.0,LEICESTERSHIRE,B-ORG,-100,IGNORE
8,##CE,10954,1,2.0,LEICESTERSHIRE,B-ORG,-100,IGNORE
9,##ST,9272,1,2.0,LEICESTERSHIRE,B-ORG,-100,IGNORE


PASS: split word, special tokens, padding, first-subtoken


In [4]:
# Minh họa nguy cơ truncation bằng câu thật và max_length=16.
row = long[0]
truncated = tokenizer(row['words'], is_split_into_words=True, truncation=True, max_length=16)
retained = {w for w in truncated.word_ids() if w is not None}
assert len(retained) < len(row['words'])
chunks = chunk_words(row['words'], tokenizer, max_length=16)
assert [i for a,b in chunks for i in range(a,b)] == list(range(len(row['words'])))
for a,b in chunks:
    part = tokenizer(row['words'][a:b], is_split_into_words=True, truncation=False)
    assert len(part['input_ids']) <= 16
    assert {w for w in part.word_ids() if w is not None} == set(range(b-a))
print('truncation retains', len(retained), '/', len(row['words']), 'words; lossless chunks:',chunks)
print('PASS: no words discarded by actual chunking strategy')

truncation retains 11 / 35 words; lossless chunks: [(0, 11), (11, 25), (25, 35)]
PASS: no words discarded by actual chunking strategy


<a id="lab09-section-16"></a>
## 16. Core Implementation

`information_extraction.encode_rows()` gọi `chunk_words()` và `align_labels()` như trên. Chunk chỉ cắt giữa hai words, không dùng truncation; một word quá dài gây lỗi rõ ràng. Prediction lấy first-subtoken argmax, ghép các chunk theo word trước BIO decoding/scoring trên câu gốc. Gold entity qua biên chunk vẫn được giữ nguyên khi score. Context hai phía biên chunk bị hạn chế; không dùng overlap để làm phức tạp bài lab. Padding động do `DataCollatorForTokenClassification` tạo, labels padding = -100; special/continuation vẫn đã được mask trước đó.


In [5]:
from information_extraction import encode_rows
records, mapping = encode_rows(data['validation'][:2], tokenizer)
batch = DataCollatorForTokenClassification(tokenizer)(records)
print({k:tuple(v.shape) for k,v in batch.items()})
assert (batch['labels'][batch['attention_mask']==0] == -100).all()
print('PASS: dynamic padding labels ignored')

{'input_ids': (2, 37), 'token_type_ids': (2, 37), 'attention_mask': (2, 37), 'labels': (2, 37)}
PASS: dynamic padding labels ignored


### Kết luận thí nghiệm tokenization
Mẫu gồm 40 câu validation đã công bố cách lấy: 20 câu đầu có <=20 words và 20 câu đầu có >20 words. Tổng 797 words thành 1.024 subwords, 116 words bị split, tăng thêm 227 mảnh. R câu ngắn=1,6023 và câu dài=1,1981; R entity=1,6036 và non-entity=1,1990. Tỷ lệ words bị split tương ứng entity=33,14%, non-entity=9,55%, khác với R. `LEICESTERSHIRE` bị chia 9 mảnh và `CRICKET` 4 mảnh. Mẫu ngắn chứa headlines viết hoa, nên kết quả không chứng minh câu ngắn nói chung bị chia nhiều hơn. First-subtoken tránh tăng trọng số loss cho một word chỉ vì word đó có nhiều mảnh; continuation vẫn cung cấp ngữ cảnh qua attention.